# 🤖 Predictive Modeling Using Machine Learning
**Thiranex Data Science Internship — Task 2**  
**Intern:** Sanjai G | **Domain:** Data Science

---
## Problem Statement
Predict whether a loan applicant will **default** on their loan using supervised machine learning models.

## Models Used
| Model | Type | Key Strength |
|---|---|---|
| Logistic Regression | Linear Classifier | Interpretable, fast baseline |
| Decision Tree | Tree-based | Visual, rule-based decisions |
| Random Forest | Ensemble | High accuracy, robust |

## Steps
1. Dataset generation (1000 loan applicants)
2. Preprocessing & feature engineering
3. Train/Test split (80/20) + feature scaling
4. Train all 3 models + 5-fold cross validation
5. Evaluate: Accuracy, ROC-AUC, Confusion Matrix, Classification Report
6. Feature Importance analysis
7. 10-chart visualization dashboard
8. Conclusion & model recommendation


In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, accuracy_score)
import warnings
warnings.filterwarnings('ignore')
plt.style.use('dark_background')
print("✅ All libraries loaded!")
print(f"scikit-learn version: {__import__('sklearn').__version__}")


In [ ]:
# ── Cell 2: Load & Explore Dataset ──────────────────────
df = pd.read_csv('loan_default_dataset.csv')
print("Shape:", df.shape)
print(f"Default Rate: {df['Default'].mean()*100:.1f}%")
print("\nMissing values:", df.isnull().sum().sum())
print("\nFirst 5 rows:")
df.head()


In [ ]:
# ── Cell 3: Statistical Summary ──────────────────────────
print("=== NUMERIC FEATURES ===")
display(df.describe().round(2))

print("\n=== TARGET CLASS DISTRIBUTION ===")
print(df['Default'].value_counts())
print(f"Class imbalance ratio: {df['Default'].value_counts()[0]/df['Default'].value_counts()[1]:.1f}:1")


In [ ]:
# ── Cell 4: Preprocessing ────────────────────────────────
df_enc = df.copy()

# Encode categorical columns
for col in ['Employment_Type', 'Education', 'Property_Area']:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    print(f"Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

X = df_enc.drop('Default', axis=1)
y = df_enc['Default']

# Train-Test Split (80/20 stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling (for Logistic Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\nTrain size: {X_train.shape} | Test size: {X_test.shape}")
print(f"Train default rate: {y_train.mean()*100:.1f}% | Test default rate: {y_test.mean()*100:.1f}%")


In [ ]:
# ── Cell 5: Train All 3 Models + Cross-Validation ────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=8,
                                                   min_samples_leaf=10, random_state=42),
}

results = {}
print(f"{'Model':<22} {'Test Acc':>10} {'CV Mean':>10} {'CV Std':>10} {'ROC-AUC':>10}")
print("─" * 65)

for name, model in models.items():
    Xtr = X_train_sc if name == 'Logistic Regression' else X_train
    Xte = X_test_sc  if name == 'Logistic Regression' else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]
    cv_scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring='accuracy')
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    results[name] = dict(model=model, y_pred=y_pred, y_prob=y_prob,
                         acc=accuracy_score(y_test, y_pred),
                         cv_mean=cv_scores.mean(), cv_std=cv_scores.std(),
                         fpr=fpr, tpr=tpr, auc=roc_auc,
                         cm=confusion_matrix(y_test, y_pred),
                         report=classification_report(y_test, y_pred, output_dict=True),
                         Xtr=Xtr, Xte=Xte)

    print(f"{name:<22} {results[name]['acc']*100:>9.2f}% {cv_scores.mean()*100:>9.2f}% "
          f"{cv_scores.std()*100:>9.2f}% {roc_auc:>10.4f}")


In [ ]:
# ── Cell 6: Classification Reports ───────────────────────
for name in models:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]['y_pred'],
                                target_names=['No Default', 'Default']))


In [ ]:
# ── Cell 7: Feature Importance (Random Forest) ───────────
rf_model = results['Random Forest']['model']
fi = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

print("=== FEATURE IMPORTANCE (Random Forest) ===")
for feat, imp in fi.items():
    bar = '█' * int(imp * 200)
    print(f"  {feat:<25} {imp:.4f}  {bar}")


In [ ]:
# ── Cell 8: Regenerate Full Dashboard ───────────────────
exec(open('ml_project.py').read())


## 📌 Results & Insights

### Model Performance Summary

| Model | Test Accuracy | CV Accuracy | ROC-AUC |
|---|---|---|---|
| Logistic Regression | 94.00% | 93.88% | 0.717 |
| Decision Tree | 94.00% | 94.12% | 0.676 |
| **Random Forest** | **94.00%** | **94.12%** | **0.701** |

### Key Findings

| # | Insight |
|---|---------|
| 1 | **Loan-to-Income Ratio** is the most important predictor (importance: 0.26) |
| 2 | **Credit Score** is the 3rd most important — lower score → higher default risk |
| 3 | **Self-Employed** applicants have a higher default rate than salaried ones |
| 4 | All models achieve ~94% accuracy due to class imbalance (only 5.9% defaulters) |
| 5 | **Logistic Regression** has the highest ROC-AUC (0.717) — best at ranking risk |
| 6 | **Random Forest** is recommended for production — most robust and stable |

## ✅ Conclusion

This project successfully built a **Loan Default Prediction** system using three supervised learning algorithms:

- **Logistic Regression**: Good probabilistic baseline, best AUC
- **Decision Tree**: Interpretable rules, visualizable for stakeholders  
- **Random Forest**: Best overall — ensemble approach reduces overfitting

**Skills demonstrated**: Data preprocessing, encoding, feature scaling, model training, cross-validation, confusion matrices, ROC curves, feature importance analysis.
